##Neste primeiro passo, procurei uma fonte confiavel para pesquisar os estados brasileiros, e escolhi como fonte de dados a api do IBGE.

In [ ]:
# Consulta na API do IBGE para verificar a forma de retorno dos dados dos estados do Brasil.

import requests
import pandas as pd

url_estados = "https://servicodados.ibge.gov.br/api/v1/localidades/estados"

response = requests.get(url_estados)
estados = response.json()

df_estados = pd.DataFrame(estados)

df_estados = df_estados[['id', 'sigla', 'nome']]
df_estados.columns = ['estado_id', 'uf', 'estado_nome']

df_estados

,estado_id,uf,estado_nome
0,11,RO,Rondônia
1,12,AC,Acre
2,13,AM,Amazonas
3,14,RR,Roraima
4,15,PA,Pará
5,16,AP,Amapá
6,17,TO,Tocantins
7,21,MA,Maranhão
8,22,PI,Piauí
9,23,CE,Ceará


##Aqui seguindo a mesma logica dos estados, fiz o mesmo para as cidades, onde consegui obter dados importantes para a construção da minha base de dados, com um dado fiel. O problema é que para obter as cordenadas, eu teria que usar Apis que são pagas, ou fazer um número inviavel de requisições na api do ibge para cada cidade.

In [15]:
# # Consulta na API do IBGE para verificar a forma de retorno das cidades dos estados do Brasil.


url_cidades = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

response = requests.get(url_cidades)
cidades = response.json()

df_cidades = pd.DataFrame(cidades)

df_cidades = pd.json_normalize(cidades)

df_cidades = df_cidades[[
    'id',
    'nome',
    'microrregiao.mesorregiao.UF.id',
    'microrregiao.mesorregiao.UF.sigla'
]]

df_cidades.columns = [
    'cidade_id',
    'cidade_nome',
    'estado_id',
    'uf'
]

# Exportando para CSV
df_cidades.to_csv('cidades_exportadas.csv', index=False, encoding='utf-8')

df_cidades

,cidade_id,cidade_nome,estado_id,uf
0,1100015,Alta Floresta D'Oeste,11.0,RO
1,1100023,Ariquemes,11.0,RO
2,1100031,Cabixi,11.0,RO
3,1100049,Cacoal,11.0,RO
4,1100056,Cerejeiras,11.0,RO
...,...,...,...,...
5566,5222005,Vianópolis,52.0,GO
5567,5222054,Vicentinópolis,52.0,GO
5568,5222203,Vila Boa,52.0,GO
5569,5222302,Vila Propício,52.0,GO


##Neste ponto eu encontrei uma base pública que parecia ser boa, importei e fiz uma análise exploratoria

In [ ]:
## https://github.com/kelvins/municipios-brasileiros$0

## E aqui eu confirmei que a base pública encontrada utilizava o mesmo id que a api do ibge retornava, e nela tinha as coordenadas que eu precisava

In [13]:
import pandas as pd

# Caminho para o arquivo CSV local
url_municipios = "/content/municipios.csv"

# Carrega o arquivo CSV diretamente em um DataFrame do pandas
df_municipios1 = pd.read_csv(url_municipios)

# Exibe as primeiras linhas do DataFrame
df_municipios1

,codigo_ibge,nome,latitude,longitude,capital,codigo_uf,siafi_id,ddd,fuso_horario
0,5200050,Abadia de Goiás,-16.75730,-49.4412,0,52,1050,62,America/Sao_Paulo
1,3100104,Abadia dos Dourados,-18.48310,-47.3916,0,31,4001,34,America/Sao_Paulo
2,5200100,Abadiânia,-16.19700,-48.7057,0,52,9201,62,America/Sao_Paulo
3,3100203,Abaeté,-19.15510,-45.4444,0,31,4003,37,America/Sao_Paulo
4,1500107,Abaetetuba,-1.72183,-48.8788,0,15,401,91,America/Sao_Paulo
...,...,...,...,...,...,...,...,...,...
5566,2517407,Zabelê,-8.07901,-37.1057,0,25,542,83,America/Sao_Paulo
5567,3557154,Zacarias,-21.05060,-50.0552,0,35,2973,18,America/Sao_Paulo
5568,2114007,Zé Doca,-3.27014,-45.6553,0,21,1287,98,America/Sao_Paulo
5569,4219853,Zortéa,-27.45210,-51.5520,0,42,950,49,America/Sao_Paulo


## Fazendo esta comparação eu pude ter certeza que minha base estava sólida e eu poderia usar as latitudes que estava nela. Salvei e importei para dentro do meu banco oracle.

In [ ]:
# Comparando os IDs das duas fontes de dados

ibge_ids = set(df_cidades['cidade_id'])
csv_ids = set(df_municipios1['codigo_ibge'])

# IDs que estão em ambas as bases
comuns = ibge_ids.intersection(csv_ids)

# IDs que estão na API mas não no CSV
apenas_api = ibge_ids - csv_ids

# IDs que estão no CSV mas não na API
apenas_csv = csv_ids - ibge_ids

print(f"Total de IDs na API do IBGE: {len(ibge_ids)}")
print(f"Total de IDs no CSV: {len(csv_ids)}")
print(f"IDs em comum: {len(comuns)}")
print(f"IDs exclusivos da API: {len(apenas_api)}")
print(f"IDs exclusivos do CSV: {len(apenas_csv)}")


Total de IDs na API do IBGE: 5571
Total de IDs no CSV: 5571
IDs em comum: 5571
IDs exclusivos da API: 0
IDs exclusivos do CSV: 0

Sucesso: Ambas as fontes possuem exatamente os mesmos IDs!
